In [26]:
import rasterio
import numpy as np
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape
import imageio
import os
from tqdm import tqdm
from PIL import Image

In [27]:
# 📂 Input raster
raster_path = "data/el_harrach_georef.tif"

# 📂 Legend folder (multiple PNGs)
legend_dir = "data/legend_area_clean"

# 📂 Output
output_dir = "output/vect/poly"
os.makedirs(output_dir, exist_ok=True)

# 🔧 tolerance for color matching
tolerance = 0

In [28]:
# -------------------------
# 🎯 DOMINANT COLOR EXTRACTOR (RGB)
# -------------------------
def get_dominant_rgb(image_path):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((100, 100))

    pixels = np.array(img).reshape(-1, 3)

    # Remove near-white background
    pixels = pixels[np.linalg.norm(pixels - [255, 255, 255], axis=1) > 30]

    if len(pixels) == 0:
        raise ValueError(f"No valid pixels found in {image_path}")

    dominant = np.median(pixels, axis=0).astype(np.uint8)
    return dominant

In [29]:
# -------------------------
# 🧠 BUILD COLOR → CLASS MAP (RGB)
# -------------------------
color_class_map = {}

print("🎨 Extracting colors from legend folder...\n")

for file in os.listdir(legend_dir):
    if not file.lower().endswith(".png"):
        continue

    path = os.path.join(legend_dir, file)
    class_name = os.path.splitext(file)[0]

    try:
        dominant_rgb = get_dominant_rgb(path)
        color_class_map[tuple(dominant_rgb)] = class_name

        print(f"{class_name}: {dominant_rgb}")

    except Exception as e:
        print(f"⚠️ Skipping {file}: {e}")

print("\n✅ Color map ready\n")

🎨 Extracting colors from legend folder...

A_place_where_you_can_skate_and_play_bandy_or_ice_hockey: [221 236 236]
Place_of_worship_where_religious_practices_are_held__other_than_building: [205 204 201]
A_military_zone_which_has_been_be_declared_to_be_dangerous_for_some_reason__i_e__a_firing_range__bombing_range__etc: [243 223 218]
Car_parking_lot__Bicycle_parking__Motorcycle_parking__Taxi_rank: [222 210 197]
National_park___Nature_reserve: [230 233 222]
Bridge: [184 184 184]
Putting_green_of_a_golf_course: [137 224 190]
Farmland___Land_area_used_for_growing_plants_in_greenhouses: [238 240 213]
Zoo: [224 223 223]
Quarry: [183 181 181]
Water_body_intermittent___Water_body_seasonal___Infiltration_basin___Detention_basin: [184 216 225]
⚠️ Skipping Area_which_belongs_to_a_kindergarten___school___college___university.png: No valid pixels found in data/legend_area_clean/Area_which_belongs_to_a_kindergarten___school___college___university.png
Allotments: [201 225 191]
Bushes_and_small_trees: 

In [30]:
# -------------------------
# 📥 READ RASTER
# -------------------------
with rasterio.open(raster_path) as src:
    img = src.read()
    transform = src.transform
    crs = src.crs

# Convert to H x W x 3 RGB image
img = np.transpose(img, (1, 2, 0))[:, :, :3].astype(np.uint8)

In [ ]:
for rgb, class_name in tqdm(color_class_map.items(), desc="Processing classes"):

    target = np.array(rgb, dtype=np.int16)

    # color mask with tolerance
    mask = np.all(np.abs(img.astype(np.int16) - target) <= tolerance, axis=2)

    # -------------------------
    # ⛔ SKIP EMPTY MASK EARLY
    # -------------------------
    if not np.any(mask):
        print(f"⚠️ Skipping {class_name}: empty mask (no pixels found)")
        continue

    mask_uint8 = mask.astype(np.uint8)

    geoms = [
        shape(geom)
        for geom, val in shapes(mask_uint8, transform=transform)
        if val == 1
    ]

    # -------------------------
    # ⛔ SKIP EMPTY GEOMETRIES
    # -------------------------
    if not geoms:
        print(f"⚠️ Skipping {class_name}: mask exists but no geometries generated")
        continue

    # -------------------------
    # 💾 SAVE GEOJSON
    # -------------------------
    gdf = gpd.GeoDataFrame(geometry=geoms, crs=crs)
    gdf["class"] = class_name

    geojson_path = os.path.join(output_dir, f"{class_name}.geojson")
    gdf.to_file(geojson_path, driver="GeoJSON")

    # -------------------------
    # 🖼️ DEBUG OVERLAY
    # -------------------------
    overlay = img.copy()

    highlight = np.zeros_like(img)
    highlight[:, :, 0] = 255

    # alpha = 0.5
    alpha = 1
    overlay[mask] = (
        (1 - alpha) * overlay[mask] + alpha * highlight[mask]
    ).astype(np.uint8)

    overlay_path = os.path.join(output_dir, f"{class_name}_overlay.png")
    imageio.imwrite(overlay_path, overlay)

print("\n✅ Done processing all classes")

Processing classes:   1%|▏         | 1/69 [00:00<00:23,  2.95it/s]

⚠️ Skipping Glacier: empty mask (no pixels found)


Processing classes:   4%|▍         | 3/69 [00:04<01:40,  1.52s/it]

⚠️ Skipping A_military_zone_which_has_been_be_declared_to_be_dangerous_for_some_reason__i_e__a_firing_range__bombing_range__etc: empty mask (no pixels found)


Processing classes:   6%|▌         | 4/69 [00:05<01:07,  1.05s/it]

...

⚠️ Skipping Commercial_area_or_business_park__predominantly_offices: empty mask (no pixels found)


Processing classes:  96%|█████████▌| 66/69 [02:39<00:02,  1.01it/s]

⚠️ Skipping Non-specific_building: empty mask (no pixels found)


Processing classes: 100%|██████████| 69/69 [02:47<00:00,  2.43s/it]

⚠️ Skipping Garages_area: empty mask (no pixels found)

✅ Done processing all classes
